# Notebook 10 — ES Arena Prospective Inference & Deployment Freeze (v0.1)

**Market Mood Forecasting · Separate Headline Arena daily experiment · 18 October 2026**

> **Research scope:** establish a separately versioned deployable ES-only three-class probability pipeline. Reuse the **frozen specification**, **not** Notebook 09's already-consumed holdout as a new test. This notebook has **no network, API credentials, Databento purchases or live submission code**.

### What this notebook does
1. Enforces the exact frozen Notebook 08 dataset/metadata contract and 12 ordered features.
2. Fits a **new future-deployment pipeline** using feature-complete data through **2026-09-30**, then verifies probability validity, fitted class order and determinism.
3. Optionally freezes a fitted `joblib` artifact, exact metadata and a SHA-256 manifest. Export requires Notebook 10 source **committed and pushed to the experimental Git branch**. Nothing is overwritten.
4. Recreates Notebook 08's previous-eligible-completed-session feature calculations from **provided and audited** completed-session aggregates; tests their lag/rolling-window semantics.
5. Optionally builds a **local, read-only Arena probability preview** from manually verified challenge and market-data files. Preview is **never** an official submission.

### Evidence limits
- Notebook 09 final: model Brier 0.656192 vs expanding prior 0.663125 across 673 sessions; paired interval includes zero; bearish argmax recall ~2.03%. **There is no established robust predictive advantage.**
- The deployment estimator here is a **new fit** that may use labels from the already-consumed 2024–2026 period, which is legitimate **only for future forecasts**, not independent backtesting.
- Historical ±0.30% labels, settlement timing and continuous contract selection may not exactly match every new Arena challenge. Fail closed on contradictions.
- Stable weekly Market Mood Forecasting **v1.2.0** and Notebook 09 are untouched.

**First local run:** leave `WRITE_DEPLOYMENT_ARTIFACTS=False` and `RUN_MANUAL_PREVIEW=False`; restart kernel → Run All. **After** reviewing and committing/pushing this exact Notebook 10, switch only `WRITE_DEPLOYMENT_ARTIFACTS=True` to create the artifact. Live preview is a separate later task.

## 1 · Immutable research configuration and safety switches

Only the two booleans identified below may be changed after committing Notebook 10; all other source changes require a new reviewed Git commit and a new deployment provenance record.

In [1]:
from pathlib import Path

EXPERIMENT = "es_arena_prospective_v0_1"
NOTEBOOK_NAME = "10_es_arena_prospective_inference_v0_1.ipynb"
FROZEN_DATA_SHA256 = "6787e64fe1c0c3508d68cd2d98cc554c6e5da19f320c7080ad7e34831b9b5db5"
NB09_PRE_HOLDOUT_COMMIT = "3b69e22d5ebe4e43029dea035c6ffef2fcf64b7c"
FROZEN_PROTOCOL_SHA256 = "396b1612edc769551f4d9be42c462be64cb88b533d95242a0a2263bbbeb6a171"
FROZEN_MODEL_SOURCE_SHA256 = "166fe687e441b6b1f0aa74659eb6fa2cfdaed72523b1380976552f777b50ae8e"
CLASSES = ("bearish", "neutral", "bullish")
TARGET_BAND_PCT = 0.30
TRAIN_THROUGH = "2026-09-30"
EXPECTED_FULL_ROWS = 3855
EXPECTED_FULL_COLS = 16
EXPECTED_FEATURE_COMPLETE_ROWS = 3835
EXPECTED_WARMUP = 20
MODEL_C = 0.01
MODEL_MAX_ITER = 2000
BASELINE_ALPHA = 1.0

# Only these two switches may be toggled without a source revision.
# TRAINING is local and safe; export needs a pushed Git-frozen Notebook 10.
WRITE_DEPLOYMENT_ARTIFACTS = False
# Read-only local preview; requires independently supplied finalized ES sessions
# AND reviewed challenge snapshot. Never calls a submission endpoint.
RUN_MANUAL_PREVIEW = False

print("Notebook 10 mode: train locally; export:", WRITE_DEPLOYMENT_ARTIFACTS,
      "; preview:", RUN_MANUAL_PREVIEW, "; network / paid data / POST: NEVER")

Notebook 10 mode: train locally; export: False ; preview: False ; network / paid data / POST: NEVER


## 2 · Environment, repository, and frozen upstream evidence

No downloads are performed. The Python environment must already include `pandas`, `numpy`, `scikit-learn`, `joblib` and a parquet engine (typically `pyarrow`). Do not install or upgrade packages inside this notebook. Model loading uses `joblib` **only for locally created trusted files**.

In [2]:
import ast
import hashlib
import io
import json
import os
import re
import subprocess
import sys
import tempfile
import warnings
from datetime import datetime, timezone, timedelta, date
from zoneinfo import ZoneInfo

import numpy as np
import pandas as pd
import sklearn
import joblib
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.exceptions import ConvergenceWarning
from IPython.display import display

UTC = timezone.utc
ET = ZoneInfo("America/New_York")
PROJECT_ROOT = next((p for p in [Path.cwd().resolve(), Path.cwd().resolve().parent]
                     if (p / "docs/headline_arena_forward_validation.md").is_file()
                     and (p / "notebooks").is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Start Jupyter inside the project or notebooks/ directory.")
NOTEBOOK_PATH = PROJECT_ROOT / "notebooks" / NOTEBOOK_NAME
PARQUET_PATH = PROJECT_ROOT / "data/processed/es_daily_modeling_v0_1.parquet"
METADATA_PATH = PROJECT_ROOT / "data/processed/es_daily_modeling_v0_1_metadata.json"
PROTOCOL_PATH = PROJECT_ROOT / "docs/headline_arena_forward_validation.md"
OUTPUT_ROOT = PROJECT_ROOT / "data/processed/es_arena_nb10_v0_1"
PREVIEW_SESSIONS_PATH = PROJECT_ROOT / "data/processed/es_arena_completed_sessions_v0_1.csv"
PREVIEW_CHALLENGE_PATH = PROJECT_ROOT / "data/processed/es_arena_challenge_review_v0_1.json"

PACKAGE_VERSIONS = {"python":sys.version.split()[0], "pandas":pd.__version__,
                    "numpy":np.__version__, "scikit-learn":sklearn.__version__,
                    "joblib":joblib.__version__}

def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(1024*1024), b""):
            h.update(block)
    return h.hexdigest()

def canonical_bytes(obj):
    return (json.dumps(obj, sort_keys=True, ensure_ascii=False,
                       separators=(",", ":"), allow_nan=False) + "\n").encode("utf-8")

def require(condition, reason):
    if not condition:
        raise RuntimeError(reason)

for label, path in [("Notebook 09 frozen parquet",PARQUET_PATH),
                    ("Notebook 08 metadata",METADATA_PATH),
                    ("frozen protocol",PROTOCOL_PATH)]:
    require(path.is_file(), f"Required {label} missing: {path}")
require(sha256_file(PARQUET_PATH) == FROZEN_DATA_SHA256,
        "Notebook 08 parquet bytes changed: STOP; do not regenerate it.")
require(sha256_file(PROTOCOL_PATH) == FROZEN_PROTOCOL_SHA256,
        "Frozen protocol changed: STOP and reconcile with Git.")
metadata_08 = json.loads(METADATA_PATH.read_text(encoding="utf-8"))
require(metadata_08["model_table_sha256"] == FROZEN_DATA_SHA256, "Metadata/parquet mismatch")
require(metadata_08["target_contract"]["target_band_pct"] == TARGET_BAND_PCT, "Target threshold drift")
require(metadata_08["target_contract"]["labels"] == list(CLASSES), "Class contract drift")
require(metadata_08["feature_contract"]["external_features_included"] is False, "External feature drift")
require(metadata_08["feature_contract"]["same_day_predeadline_features_included"] is False,
        "Same-day features introduced")
FEATURES = tuple(metadata_08["feature_contract"]["feature_columns"])
EXPECTED_FEATURES = (
    "es_return_lag1","es_return_lag2","es_return_lag3","es_return_lag5",
    "es_range_pct_lag1","es_log_volume_lag1","es_return_mean_5",
    "es_return_mean_20","es_vol_5","es_vol_20",
    "es_abs_return_mean_5","es_abs_return_mean_20")
require(FEATURES == EXPECTED_FEATURES, "Feature names or feature ORDER changed")
print("PROJECT:", PROJECT_ROOT)
print("Python:",sys.executable)
print("Versions:",PACKAGE_VERSIONS)
print("FROZEN UPSTREAM CONTRACT: PASS; all 12 feature names and order verified")

PROJECT: C:\Users\melny\Market-Mood-Forecasting
Python: C:\Users\melny\Market-Mood-Forecasting\.venv\Scripts\python.exe
Versions: {'python': '3.12.10', 'pandas': '3.0.6', 'numpy': '2.5.3', 'scikit-learn': '1.9.1', 'joblib': '1.6.0'}
FROZEN UPSTREAM CONTRACT: PASS; all 12 feature names and order verified


## 3 · Verify the modeling table without rerunning Notebook 09

Unlike Notebook 09, **no final-test metrics or hyperparameter search** is performed here. Using all past data to fit a *new future-deployment model* is allowed, but its fitted coefficients are not Notebook 09's quarterly fold coefficients.

In [3]:
try:
    table = pd.read_parquet(PARQUET_PATH)
except ImportError as e:
    raise ImportError("Your existing .venv needs its parquet engine (usually pyarrow). "
                      "Install it from Command Prompt into the .venv; do not change this notebook.") from e
require(table.shape == (EXPECTED_FULL_ROWS,EXPECTED_FULL_COLS),
        f"Frozen dataframe shape changed: {table.shape}")
require(tuple(table.columns) == ("session_date","target_return","target","target_band_pct")+FEATURES,
        "Saved modeling-table column order drift")
train_all = table.copy()
train_all["session_date"] = pd.to_datetime(train_all["session_date"],errors="raise")
require(train_all.session_date.dt.tz is None, "Expected date-only session_date")
require(train_all.session_date.is_monotonic_increasing and not train_all.session_date.duplicated().any(),
        "Session dates must be ordered and unique")
require(str(train_all.session_date.iloc[0].date()) == "2010-06-08", "First session changed")
require(str(train_all.session_date.iloc[-1].date()) == TRAIN_THROUGH,"Frozen train cutoff changed")
require(train_all.target.notna().all() and set(train_all.target)==set(CLASSES), "Target labels missing/changed")
returns = pd.to_numeric(train_all.target_return,errors="raise").to_numpy(float)
require(np.isfinite(returns).all(), "Non-finite historical target returns")
expected_target = np.where(returns > TARGET_BAND_PCT/100,"bullish",
                           np.where(returns < -TARGET_BAND_PCT/100,"bearish","neutral"))
require(np.array_equal(expected_target,train_all.target.astype(str).to_numpy()),"Frozen labels inconsistent")
require(np.allclose(train_all.target_band_pct.astype(float),TARGET_BAND_PCT,atol=1e-12),"Band drift")
X_source = train_all.loc[:,FEATURES].apply(pd.to_numeric,errors="raise")
values = X_source.to_numpy(dtype=float,na_value=np.nan)
require(not np.isinf(values).any(),"Infinite feature values")
missing = ~np.isfinite(values).all(axis=1)
require(missing.sum()==EXPECTED_WARMUP and missing[:EXPECTED_WARMUP].all()
        and not missing[EXPECTED_WARMUP:].any(),"Unexpected feature warm-up / missing data")
training = train_all.loc[~missing].copy().reset_index(drop=True)
require(len(training)==EXPECTED_FEATURE_COMPLETE_ROWS,"Unexpected training sample size")
require(training.session_date.max()==pd.Timestamp(TRAIN_THROUGH),"Training cutoff mismatch")
require(training["session_date"].is_unique,"Training dates not unique")
X_train = training.loc[:,FEATURES].astype(float)
y_train = training.target.astype(str)
require(set(y_train)==set(CLASSES),"Training must contain all classes")
print("HISTORICAL TABLE GATE: PASS")
print("Model-fitting rows:",len(training),"; cutoff:",TRAIN_THROUGH)
print("This is NEW DEPLOYMENT TRAINING, not a second independent holdout evaluation.")

HISTORICAL TABLE GATE: PASS
Model-fitting rows: 3835 ; cutoff: 2026-09-30
This is NEW DEPLOYMENT TRAINING, not a second independent holdout evaluation.


## 4 · Fit the future-deployment estimator (no retuning)

Exactly the model **family and parameters** frozen in Notebook 09: `StandardScaler → LogisticRegression(C=0.01, penalty='l2', solver='lbfgs', class_weight=None, max_iter=2000)`. The scaler is fitted only inside the training pipeline. We compute **no new historical test score** and optimize nothing.

In [4]:
def make_deployment_pipeline():
    return Pipeline([
        ("scale",StandardScaler()),
        ("logreg",LogisticRegression(C=MODEL_C,solver="lbfgs",penalty="l2",
                                     class_weight=None,max_iter=MODEL_MAX_ITER))
    ])

model = make_deployment_pipeline()
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always",ConvergenceWarning)
    model.fit(X_train,y_train)
require(not any(issubclass(w.category,ConvergenceWarning) for w in caught),
        "Logistic regression convergence warning: STOP, record it; do not silently retune")
NATIVE_CLASSES = tuple(model.named_steps["logreg"].classes_)
require(set(NATIVE_CLASSES)==set(CLASSES),"Trained class set drift")


def check_probability_matrix(p, name="probabilities"):
    a = np.asarray(p,dtype=float)
    require(a.ndim == 2 and a.shape[1]==3, f"{name} must have shape (n,3)")
    require(np.isfinite(a).all(), f"Nonfinite {name}")
    require((a >= 0).all() and (a<=1).all(), f"{name} outside [0,1]")
    require(np.allclose(a.sum(axis=1),1.0,atol=1e-10,rtol=0),f"{name} do not sum to one")
    return a


def ordered_model_probabilities(fitted_model, feature_df):
    require(list(feature_df.columns)==list(FEATURES),"Wrong feature order")
    arr = feature_df.to_numpy(dtype=float)
    require(np.isfinite(arr).all(),"Inference contains missing/invalid features")
    native = tuple(fitted_model.named_steps["logreg"].classes_)
    require(set(native)==set(CLASSES),"Model class identity mismatch")
    p_native = fitted_model.predict_proba(feature_df)
    return check_probability_matrix(p_native[:,[native.index(c) for c in CLASSES]])

probe_features = X_train.iloc[[0,-1]].copy()
probe_p = ordered_model_probabilities(model,probe_features)
assert np.allclose(probe_p,ordered_model_probabilities(model,probe_features),atol=0,rtol=0)

# Prior uses ALL 3,855 eligible settled labels (incl. 20 model warm-up sessions);
# held fixed as deployment benchmark until separately re-versioned.
prior_counts = train_all.target.value_counts().reindex(CLASSES,fill_value=0).astype(int)
prior_p = (prior_counts.to_numpy(dtype=float) + BASELINE_ALPHA) / (len(train_all)+3*BASELINE_ALPHA)
check_probability_matrix(prior_p.reshape(1,3),"frozen training prior")
print("DEPLOYMENT ESTIMATOR FIT: PASS")
print("Native classes:",NATIVE_CLASSES,"; output classes:",CLASSES)
print("Engineering smoke test: probabilities finite, normalized, repeatable")
print("Static prior from",len(train_all),"settled labels (not Notebook 09's daily-refreshed prior)")

DEPLOYMENT ESTIMATOR FIT: PASS
Native classes: ('bearish', 'bullish', 'neutral') ; output classes: ('bearish', 'neutral', 'bullish')
Engineering smoke test: probabilities finite, normalized, repeatable
Static prior from 3855 settled labels (not Notebook 09's daily-refreshed prior)


## 5 · Notebook source identity, Git handoff, and artifact write-once controls

The notebook must be saved under the canonical filename. **To export**: commit & push the notebook first (with both switches `False`), then toggle only `WRITE_DEPLOYMENT_ARTIFACTS=True` and run all. The source comparison deliberately ignores **only** the two approved switch values, not arbitrary edits. A model artifact is stored under ignored `data/processed/`, never automatically staged or uploaded. Existing artifacts are verified rather than overwritten.

**Upstream provenance diagnostic:** If the Notebook 09 Git source SHA does not match the frozen approval hash, Notebook 10 prints the expected SHA, the observed Git SHA, and any different Git-vs-local source-cell indices. This is **not a model-training error**. Training-only checks may continue, but **model artifact export and preview remain blocked** until the mismatch is reconciled; do not change the frozen Notebook 09 code/hash just to make this check pass.


In [5]:
def git_read(*args):
    p = subprocess.run(["git",*args],cwd=PROJECT_ROOT,text=True,encoding="utf-8",errors="strict",capture_output=True)
    if p.returncode:
        raise RuntimeError(f"Git {' '.join(args)} failed: {p.stderr.strip()}")
    return p.stdout.strip()

GIT_HEAD = git_read("rev-parse","HEAD")
GIT_BRANCH = git_read("branch","--show-current")
require(GIT_BRANCH == "experiment/headline-arena-es-daily",
        "Wrong Git branch — never work on stable main")

# Independently reproduce Notebook 09's source-only fingerprint from the
# immutable pre-holdout Git object. This checks the actual parent research,
# not just a hard-coded Git commit reference or notebook filename.
def frozen_nb09_source_fingerprint(doc):
    seen={"RUN_LOCKED_FINAL_TEST":0,"FINAL_TEST_CONFIRMATION":0}
    sources=[]
    for i,c in enumerate(doc["cells"]):
        text=''.join(c['source'])
        if c['cell_type']=='code':
            tree=ast.parse(text)
            rows=text.splitlines(keepends=True)
            edits=[]
            for node in ast.walk(tree):
                if not isinstance(node,ast.Assign):
                    continue
                targets=[x.id for x in node.targets if isinstance(x,ast.Name) and x.id in seen]
                if targets:
                    require(len(node.targets)==1 and len(targets)==1 and node in tree.body
                            and isinstance(node.value,ast.Constant) and node.lineno==node.end_lineno,
                            "Unsafe Notebook 09 authorization source")
                    name=targets[0]; seen[name]+=1
                    target_value = False if name=="RUN_LOCKED_FINAL_TEST" else ""
                    require(type(node.value.value) is type(target_value),
                            "Unexpected Notebook 09 authorization literal type")
                    trailing=rows[node.end_lineno-1][node.end_col_offset:].strip()
                    require(not trailing or trailing.startswith('#'),
                            "Notebook 09 unauthorized code appended on a switch line")
                    edits.append((node.value.lineno-1,node.value.col_offset,
                                  node.value.end_col_offset,repr(target_value)))
            for row,start,end,replacement in sorted(edits,reverse=True):
                rows[row]=rows[row][:start]+replacement+rows[row][end:]
            text=''.join(rows)
        sources.append({"cell_type":c['cell_type'],"source":text})
    require(seen=={"RUN_LOCKED_FINAL_TEST":1,"FINAL_TEST_CONFIRMATION":1},
            "Notebook 09 approval source assignments missing/duplicated")
    return hashlib.sha256(json.dumps(sources,sort_keys=True,ensure_ascii=False,
                                   separators=(',',':')).encode('utf-8')).hexdigest()

# Upstream provenance diagnosis — read-only. Never silently substitute a
# different source hash, commit, or local file when Git does not agree.
nb09_original=json.loads(git_read("show",f"{NB09_PRE_HOLDOUT_COMMIT}:notebooks/09_es_daily_walkforward_modeling.ipynb"))
OBSERVED_NB09_GIT_SHA256 = frozen_nb09_source_fingerprint(nb09_original)
UPSTREAM_GIT_VERIFIED = OBSERVED_NB09_GIT_SHA256 == FROZEN_MODEL_SOURCE_SHA256
print("EXPECTED NOTEBOOK 09 SOURCE SHA-256:", FROZEN_MODEL_SOURCE_SHA256)
print("GIT COMMIT NOTEBOOK 09 SOURCE SHA-256:", OBSERVED_NB09_GIT_SHA256)
print("GIT COMMIT:",NB09_PRE_HOLDOUT_COMMIT)
print("FROZEN NOTEBOOK 09 GIT SOURCE FINGERPRINT:",
      "PASS" if UPSTREAM_GIT_VERIFIED else "FAIL — upstream provenance needs investigation")
if not UPSTREAM_GIT_VERIFIED:
    # This diagnostic neither reads the consumed test outcomes nor modifies
    # any research files. It only compares normalized cell source fingerprints.
    git_sources = [(c["cell_type"], "".join(c["source"]))
                   for c in nb09_original["cells"]]
    nb09_local_path=PROJECT_ROOT/"notebooks/09_es_daily_walkforward_modeling.ipynb"
    if nb09_local_path.is_file():
        nb09_local=json.loads(nb09_local_path.read_text(encoding="utf-8-sig"))
        local_hash=frozen_nb09_source_fingerprint(nb09_local)
        print("LOCAL RESTORED NOTEBOOK 09 SOURCE SHA-256:",local_hash)
        local_sources=[(c["cell_type"], "".join(c["source"]))
                       for c in nb09_local["cells"]]
        source_differences=[i for i in range(max(len(git_sources),len(local_sources)))
                            if i>=len(git_sources) or i>=len(local_sources)
                            or git_sources[i]!=local_sources[i]]
        print("GIT-vs-LOCAL SOURCE CELL DIFFERENCES (indices only):",source_differences[:40])
    else:
        print("Local canonical Notebook 09 unavailable; cannot compare local source")
    if WRITE_DEPLOYMENT_ARTIFACTS or RUN_MANUAL_PREVIEW:
        raise RuntimeError(
            "STOP: frozen Notebook 09 Git source hash mismatch. "
            "Artifact export and preview are prohibited until the research "
            "provenance conflict is resolved. Do NOT edit or amend Notebook 09 "
            "or replace its hard-coded frozen hash. Share the three hashes "
            "and changed source-cell indices shown above for diagnosis.")
    print("WARNING: TRAINING-ONLY MODE. Continuing synthetic and model checks, "
          "but refusing all model exports until Notebook 09 provenance is reconciled.")

_SWITCHES = {"WRITE_DEPLOYMENT_ARTIFACTS","RUN_MANUAL_PREVIEW"}

def canonical_code_sources(notebook_payload):
    sources = []
    for cell in notebook_payload["cells"]:
        if cell["cell_type"]!="code":
            continue
        source = ''.join(cell["source"])
        tree = ast.parse(source)
        for node in ast.walk(tree):
            if isinstance(node,(ast.Assign,ast.AnnAssign)):
                targets = node.targets if isinstance(node,ast.Assign) else [node.target]
                if any(isinstance(t,ast.Name) and t.id in _SWITCHES for t in targets):
                    if len(targets)!=1 or not isinstance(targets[0],ast.Name) or not isinstance(node.value,ast.Constant) or type(node.value.value) is not bool:
                        raise RuntimeError("Approval switch must be an isolated literal boolean assignment")
                    node.value.value = False
        sources.append(ast.dump(tree,annotate_fields=True,include_attributes=False))
    return sources

require(NOTEBOOK_PATH.is_file(),f"Save the notebook as canonical path before running: {NOTEBOOK_PATH}")
notebook_local = json.loads(NOTEBOOK_PATH.read_text(encoding="utf-8"))
NORMALIZED_CODE = canonical_code_sources(notebook_local)
NOTEBOOK_CODE_HASH = hashlib.sha256(canonical_bytes(NORMALIZED_CODE)).hexdigest()


def assert_fresh_jupyter_execution_through(marker):
    """Require a fresh saved-source, in-order Jupyter Run All before write-once actions.

    This is accidental-drift protection; it cannot defeat intentional tampering
    with the Python process or filesystem.
    """
    from IPython import get_ipython
    ip=get_ipython()
    require(ip is not None and hasattr(ip,'history_manager'),
            "Use a real Jupyter kernel for artifact export / live preview")
    saved=json.loads(NOTEBOOK_PATH.read_text(encoding='utf-8'))
    code_cells=[''.join(c['source']) for c in saved['cells'] if c['cell_type']=='code']
    matched=[i for i,source in enumerate(code_cells) if marker in source]
    require(len(matched)==1,"Write-once action sentinel missing or duplicated")
    expected=code_cells[:matched[0]+1]
    actual=list(ip.history_manager.input_hist_raw[1:])
    norm=lambda x:x.replace('\r\n','\n').rstrip('\n')
    require(len(actual)==len(expected) and [norm(v) for v in actual]==[norm(v) for v in expected],
            "Kernel is not a fresh sequential Run All matching the SAVED notebook. "
            "Save, Restart Kernel, then Run All. Do not run cells out of order.")
    require(canonical_code_sources(saved)==NORMALIZED_CODE,
            "Notebook source changed after initial fingerprinting")
    print("FRESH SAVED-VS-EXECUTED JUPYTER SOURCE: PASS")

try:
    canonical_git = json.loads(git_read("show",f"HEAD:notebooks/{NOTEBOOK_NAME}"))
    NOTEBOOK_COMMITTED = (NORMALIZED_CODE==canonical_code_sources(canonical_git))
except RuntimeError:
    NOTEBOOK_COMMITTED = False
try:
    REMOTE_HASH = git_read("rev-parse","origin/experiment/headline-arena-es-daily")
except RuntimeError:
    REMOTE_HASH = ""
GIT_SYNCED = REMOTE_HASH == GIT_HEAD

print("SOURCE SHA-256 (normalized code):", NOTEBOOK_CODE_HASH)
print("Notebook 10 matches committed code (two safe switches normalized):", NOTEBOOK_COMMITTED)
print("Pushed branch synchronized:",GIT_SYNCED)
if not NOTEBOOK_COMMITTED:
    print("NEXT: review and commit Notebook 10 before enabling export.")
if not GIT_SYNCED:
    print("NEXT: git push and git fetch before enabling export.")

EXPECTED NOTEBOOK 09 SOURCE SHA-256: 166fe687e441b6b1f0aa74659eb6fa2cfdaed72523b1380976552f777b50ae8e
GIT COMMIT NOTEBOOK 09 SOURCE SHA-256: 166fe687e441b6b1f0aa74659eb6fa2cfdaed72523b1380976552f777b50ae8e
GIT COMMIT: 3b69e22d5ebe4e43029dea035c6ffef2fcf64b7c
FROZEN NOTEBOOK 09 GIT SOURCE FINGERPRINT: PASS
SOURCE SHA-256 (normalized code): 42164bb1a566165f0cadddd0791f6f91d8dd3e4c0310c9359deb20f08d17c9d0
Notebook 10 matches committed code (two safe switches normalized): False
Pushed branch synchronized: True
NEXT: review and commit Notebook 10 before enabling export.


In [6]:
# NB10_EXPORT_GATE_SENTINEL_V0_1
TRAINING_IDENTITY = {
    "experiment":EXPERIMENT,
    "frozen_notebook_09_commit":NB09_PRE_HOLDOUT_COMMIT,
    "notebook10_source_sha256":NOTEBOOK_CODE_HASH,
    "git_commit_at_fit":GIT_HEAD,
    "dataset_sha256":FROZEN_DATA_SHA256,
    "protocol_sha256":FROZEN_PROTOCOL_SHA256,
    "train_through":TRAIN_THROUGH,
    "feature_columns":list(FEATURES),
    "class_order":list(CLASSES),
    "model_C":MODEL_C,
    "model_params":{"solver":"lbfgs","penalty":"l2","class_weight":None,
                    "max_iter":MODEL_MAX_ITER,"scale":"StandardScaler"},
    "packages":PACKAGE_VERSIONS,
}
DEPLOYMENT_ID = hashlib.sha256(canonical_bytes(TRAINING_IDENTITY)).hexdigest()[:16]
MODEL_DIR = OUTPUT_ROOT / f"deployment_{DEPLOYMENT_ID}"
MODEL_NAME = "es_daily_lr_deployment_v0_1.joblib"
MODEL_METADATA_NAME = "es_daily_lr_deployment_v0_1_metadata.json"
MODEL_MANIFEST_NAME = "es_daily_lr_deployment_v0_1_manifest.json"


def deployment_metadata():
    return {
        "deployment_identity":TRAINING_IDENTITY,
        "model_training_rows":len(training),
        "all_target_eligible_labels_for_prior":len(train_all),
        "train_start_date":str(training.session_date.min().date()),
        "train_end_date":str(training.session_date.max().date()),
        "label_counts_training":{c:int((y_train==c).sum()) for c in CLASSES},
        "static_prior_label_counts":{c:int(prior_counts[c]) for c in CLASSES},
        "static_prior_probabilities":{c:float(prior_p[i]) for i,c in enumerate(CLASSES)},
        "native_estimator_classes":list(NATIVE_CLASSES),
        "status":"NEW FUTURE DEPLOYMENT FIT; NOT NB09 WALK-FORWARD ESTIMATOR",
        "holdout_2024_2026":"Previously consumed as Notebook 09 independent evaluation; used here only for future training",
        "historical_validation":"NB09 small numerical advantage, CI includes zero; bearish argmax recall ~2%",
        "submission_policy":"No network or POST; prediction requires manual review",
        "model_input_feature_dtype":"float64",
        "saved_probe_probabilities":probe_p.tolist(),
    }


def validate_saved_model(artifact_dir):
    artifact_dir = Path(artifact_dir)
    paths = [artifact_dir/MODEL_NAME,artifact_dir/MODEL_METADATA_NAME,
             artifact_dir/MODEL_MANIFEST_NAME]
    require(all(p.is_file() for p in paths),"Partial existing deployment artifact: refuse overwrite")
    saved_manifest = json.loads(paths[2].read_text(encoding="utf-8"))
    require(saved_manifest["model_sha256"] == sha256_file(paths[0]),"Joblib checksum drift")
    require(saved_manifest["metadata_sha256"] == sha256_file(paths[1]),"Metadata checksum drift")
    saved_metadata = json.loads(paths[1].read_text(encoding="utf-8"))
    require(saved_metadata == deployment_metadata(),"Frozen deployment metadata drift")
    # SECURITY: joblib can execute arbitrary code. Load ONLY OUR OWN freshly created local file.
    loaded = joblib.load(paths[0])
    new_probs = ordered_model_probabilities(loaded,probe_features)
    require(np.allclose(new_probs,probe_p,atol=1e-12,rtol=0),"Reloaded model probability mismatch")
    return loaded,saved_manifest


def create_or_verify_deployment():
    require(UPSTREAM_GIT_VERIFIED,
            "Cannot export: frozen Notebook 09 Git source fingerprint is not verified")
    require(NOTEBOOK_COMMITTED and GIT_SYNCED,"Commit/push Notebook 10 source before artifact export")
    if MODEL_DIR.exists():
        loaded,manifest = validate_saved_model(MODEL_DIR)
        print("WRITE-ONCE DEPLOYMENT: existing verified (no files overwritten)")
        return loaded,manifest
    OUTPUT_ROOT.mkdir(parents=True,exist_ok=True)
    # Stage files on the same filesystem, then rename the entire directory atomically.
    with tempfile.TemporaryDirectory(prefix=".nb10_stage_",dir=OUTPUT_ROOT) as tmp:
        stage = Path(tmp)
        joblib.dump(model,stage/MODEL_NAME,compress=3)
        (stage/MODEL_METADATA_NAME).write_bytes(canonical_bytes(deployment_metadata()))
        manifest = {
            "deployment_id":DEPLOYMENT_ID,
            "model_sha256":sha256_file(stage/MODEL_NAME),
            "metadata_sha256":sha256_file(stage/MODEL_METADATA_NAME),
            "commit":GIT_HEAD,
            "dataset_sha256":FROZEN_DATA_SHA256,
            "source_sha256":NOTEBOOK_CODE_HASH,
            "written_utc":datetime.now(UTC).isoformat(),
            "write_once":True,
        }
        (stage/MODEL_MANIFEST_NAME).write_bytes(canonical_bytes(manifest))
        require(not MODEL_DIR.exists(),"Target unexpectedly appeared; stop, do not overwrite")
        # On Windows os.rename refuses an existing destination.
        os.rename(stage,MODEL_DIR)
    loaded,saved_manifest = validate_saved_model(MODEL_DIR)
    print("WRITE-ONCE DEPLOYMENT: CREATED and VERIFIED:",MODEL_DIR)
    return loaded,saved_manifest

FROZEN_MODEL = None
FROZEN_MANIFEST = None
if WRITE_DEPLOYMENT_ARTIFACTS:
    assert_fresh_jupyter_execution_through("# NB10_EXPORT_GATE_SENTINEL_V0_1")
    FROZEN_MODEL,FROZEN_MANIFEST = create_or_verify_deployment()
else:
    print("SAFE TRAINING-ONLY RUN: no deployment files created")
    if MODEL_DIR.is_dir():
        print("Pre-existing model directory noted; no loading without explicit export check")
print("Deployment ID:",DEPLOYMENT_ID)
print("Expected model directory:",MODEL_DIR)

SAFE TRAINING-ONLY RUN: no deployment files created
Deployment ID: 4e649892f59759c1
Expected model directory: C:\Users\melny\Market-Mood-Forecasting\data\processed\es_arena_nb10_v0_1\deployment_4e649892f59759c1


## 6 · Derive EXACT previous-eligible-session features from completed aggregates

Notebook 08 definition: use **only valid, target-eligible, finished sessions**. Lags are `1,2,3,5`; trailing return means and sample standard deviations use `5` and `20` eligible observations; volume is `log1p(volume)`; range is `(high − low)/open`. Excluded sessions **never** enter feature history. This code contains no provider calls.

**Live CSV schema** (one row per completed settlement-day ES session, including excluded rows): `session_date`, `session_start_utc`, `session_end_utc`, `data_available_utc`, `open`, `high`, `low`, `close`, `volume`, `session_return`, `raw_symbol`, `target_eligible`, `first_bar_on_expected_boundary`, `last_bar_on_expected_boundary`, `multiple_underlying_symbols`, `coverage_reviewed`, `source_dataset`, `source_schema`.

All timestamps must be UTC with offsets. `session_date` is the **ET settlement date**, not the 18:00 ET opening date. The CSV must contain the **consecutive reviewed session opportunities**, not just a cherry-picked subset of days. Missing source days are not automatically detectable: a human coverage attestation remains mandatory.

In [7]:
REQUIRED_SESSION_COLUMNS = (
    "session_date","session_start_utc","session_end_utc","data_available_utc",
    "open","high","low","close","volume","session_return","raw_symbol",
    "target_eligible","first_bar_on_expected_boundary","last_bar_on_expected_boundary",
    "multiple_underlying_symbols","coverage_reviewed","source_dataset","source_schema")

def bool_column(series, name):
    tokens = series.astype(str).str.strip().str.lower()
    require(tokens.isin(["true","false","1","0"]).all(),f"{name} contains invalid/missing booleans")
    return tokens.isin(["true","1"])


def as_utc_timestamp(value, label):
    ts = pd.Timestamp(value)
    require(ts.tzinfo is not None, f"{label} must contain an explicit UTC offset")
    return ts.tz_convert("UTC")


def session_time_utc(d, clock_hour, prior_day=False):
    local = pd.Timestamp(d).date()
    if prior_day:
        local -= timedelta(days=1)
    return pd.Timestamp(datetime(local.year,local.month,local.day,clock_hour,
                                 tzinfo=ET)).tz_convert("UTC")


def validate_session_history(csv_or_df, challenge_session_date, information_cutoff_utc):
    raw = pd.read_csv(csv_or_df) if isinstance(csv_or_df,(str,Path)) else csv_or_df.copy()
    require(set(REQUIRED_SESSION_COLUMNS).issubset(raw.columns),
            f"Session source is missing columns: {sorted(set(REQUIRED_SESSION_COLUMNS)-set(raw.columns))}")
    require(len(raw)>=20,"At least 20 completed eligible source sessions are required")
    x=raw.loc[:,REQUIRED_SESSION_COLUMNS].copy()
    x["session_date"]=pd.to_datetime(x.session_date,errors="raise")
    require(x.session_date.dt.tz is None,"Session dates must be ET date-only")
    require(x.session_date.dt.normalize().eq(x.session_date).all(),"Session date has a time component")
    require(x.session_date.is_monotonic_increasing and x.session_date.is_unique,
            "Sessions must be strictly increasing and unique")
    future_day=pd.Timestamp(challenge_session_date)
    require(future_day == future_day.normalize(),"Challenge settlement date must be date-only")
    require((x.session_date < future_day).all(),"Same-day/future sessions: LEAKAGE BLOCKED")
    cutoff=as_utc_timestamp(information_cutoff_utc,"information_cutoff_utc")
    for field in ["target_eligible","first_bar_on_expected_boundary",
                  "last_bar_on_expected_boundary","multiple_underlying_symbols","coverage_reviewed"]:
        x[field] = bool_column(x[field],field)
    require(x.coverage_reviewed.all(),"All session rows need coverage-reviewed attestation")
    require(x.source_dataset.astype(str).str.strip().ne("").all() and
            x.source_schema.astype(str).str.strip().ne("").all(),"Missing source provenance")
    expected_eligibility=(x.first_bar_on_expected_boundary
                          & x.last_bar_on_expected_boundary
                          & ~x.multiple_underlying_symbols)
    require((~x.target_eligible | expected_eligibility).all(),
            "A source marked eligible fails Notebook 08 boundary/contract flags")
    eligible=x.loc[x.target_eligible].copy()
    require(len(eligible)>=20,"Need 20 eligible completed source sessions")
    for field in ["open","high","low","close","volume","session_return"]:
        eligible[field]=pd.to_numeric(eligible[field],errors="raise")
    numeric=eligible[["open","high","low","close","volume","session_return"]].to_numpy(float)
    require(np.isfinite(numeric).all(),"Eligible source has non-finite prices, return or volume")
    require((eligible[["open","high","low","close"]]>0).all().all(),"Non-positive ES price")
    require((eligible.high>=eligible[["open","close","low"]].max(axis=1)).all() and
            (eligible.low<=eligible[["open","close","high"]].min(axis=1)).all(),
            "OHLC range inconsistent")
    require((eligible.volume>=0).all(),"Negative volume")
    require(np.allclose(eligible.session_return.to_numpy(float),
                        eligible.close.to_numpy(float)/eligible.open.to_numpy(float)-1,
                        rtol=0,atol=1e-8),"Session return differs from close/open - 1")
    require(eligible.raw_symbol.astype(str).str.match(r"^ES[HMUZ][0-9]{1,2}$").all(),
            "Underlying symbol must be a concrete ES futures contract")
    for _,row in eligible.iterrows():
        start=as_utc_timestamp(row.session_start_utc,"session_start_utc")
        end=as_utc_timestamp(row.session_end_utc,"session_end_utc")
        available=as_utc_timestamp(row.data_available_utc,"data_available_utc")
        d=row.session_date
        require(start==session_time_utc(d,18,prior_day=True),"Expected prior-day 18:00 ET opening")
        require(end==session_time_utc(d,17),"Expected settlement-day 17:00 ET end")
        require(start<end<=available<=cutoff,
                "Incomplete/unavailable source session or data arrival AFTER cutoff")
    require(eligible.session_date.max()<future_day,"Future eligible session detected")
    # No silent use of September 30 when the current target is already in mid-October.
    gap=(future_day.date()-eligible.session_date.max().date()).days
    require(0<gap<=7,"Last eligible source is stale (>7 calendar days): update data first")
    return eligible.reset_index(drop=True)


def feature_state_from_eligible_sessions(eligible):
    require(len(eligible)>=20,"20 eligible sessions required")
    end=eligible.tail(20)
    r=end.session_return.to_numpy(dtype=float)
    latest=eligible.iloc[-1]
    feats={
        "es_return_lag1":float(r[-1]),
        "es_return_lag2":float(r[-2]),
        "es_return_lag3":float(r[-3]),
        "es_return_lag5":float(r[-5]),
        "es_range_pct_lag1":float((latest.high-latest.low)/latest.open),
        "es_log_volume_lag1":float(np.log1p(latest.volume)),
        "es_return_mean_5":float(np.mean(r[-5:])),
        "es_return_mean_20":float(np.mean(r)),
        "es_vol_5":float(np.std(r[-5:],ddof=1)),
        "es_vol_20":float(np.std(r,ddof=1)),
        "es_abs_return_mean_5":float(np.mean(np.abs(r[-5:]))),
        "es_abs_return_mean_20":float(np.mean(np.abs(r))),
    }
    require(set(feats)==set(FEATURES),"Feature implementation mismatch")
    out=pd.DataFrame([[feats[k] for k in FEATURES]],columns=FEATURES,dtype=float)
    require(np.isfinite(out.to_numpy(dtype=float)).all(),"Nonfinite computed live feature")
    return out

### Independent synthetic as-of and exclusion tests (NO historical holdout reused)

Tests check numeric rolling mathematics, selected lag semantics, excluded sessions, day boundaries and fail-closed timestamp behavior. They use **artificial sessions only**; they do not establish live data-quality coverage.

In [8]:
def synthetic_sessions():
    dts=pd.bdate_range("2026-07-01",periods=28)
    out=[]
    for i,d in enumerate(dts):
        op=5500.0+i; ret=(i-13)/10000.0
        cl=op*(1+ret)
        out.append({"session_date":str(d.date()),
                    "session_start_utc":session_time_utc(d,18,True).isoformat(),
                    "session_end_utc":session_time_utc(d,17).isoformat(),
                    "data_available_utc":(session_time_utc(d,17)+pd.Timedelta(minutes=30)).isoformat(),
                    "open":op,"high":max(op,cl)+2,"low":min(op,cl)-2,"close":cl,
                    "volume":100000+i*5,"session_return":ret,"raw_symbol":"ESU6",
                    "target_eligible":True,"first_bar_on_expected_boundary":True,
                    "last_bar_on_expected_boundary":True,"multiple_underlying_symbols":False,
                    "coverage_reviewed":True,"source_dataset":"SYNTHETIC_TEST_ONLY",
                    "source_schema":"synthetic_complete_session"})
    return pd.DataFrame(out)

_synth=synthetic_sessions()
_last_date=pd.Timestamp(_synth.session_date.iloc[-1]); _future_date=str((_last_date+pd.Timedelta(days=1)).date())
_asof=(session_time_utc(_last_date,17)+pd.Timedelta(hours=2)).isoformat()
_good=validate_session_history(_synth,_future_date,_asof)
_vec=feature_state_from_eligible_sessions(_good)
_source_returns=_good.session_return.to_numpy(float)
assert np.isclose(_vec.es_return_lag5.iloc[0],_source_returns[-5])
assert np.isclose(_vec.es_return_mean_20.iloc[0],np.mean(_source_returns[-20:]))
assert np.isclose(_vec.es_vol_20.iloc[0],np.std(_source_returns[-20:],ddof=1))
assert np.isclose(_vec.es_log_volume_lag1.iloc[0],np.log1p(_good.volume.iloc[-1]))
assert np.isclose(_vec.es_range_pct_lag1.iloc[0],
                  (_good.high.iloc[-1]-_good.low.iloc[-1])/_good.open.iloc[-1])
# Independent pandas shift/rolling calculation: exactly the Notebook 08 formula
# before mapping the state to a STRICTLY LATER forecast date.
_r=pd.Series(_good.session_return.to_numpy(float))
_ref={"es_return_lag1":_r.iloc[-1],"es_return_lag2":_r.shift(1).iloc[-1],
      "es_return_lag3":_r.shift(2).iloc[-1],"es_return_lag5":_r.shift(4).iloc[-1],
      "es_range_pct_lag1":(_good.high.iloc[-1]-_good.low.iloc[-1])/_good.open.iloc[-1],
      "es_log_volume_lag1":np.log1p(_good.volume.iloc[-1]),
      "es_return_mean_5":_r.rolling(5,min_periods=5).mean().iloc[-1],
      "es_return_mean_20":_r.rolling(20,min_periods=20).mean().iloc[-1],
      "es_vol_5":_r.rolling(5,min_periods=5).std().iloc[-1],
      "es_vol_20":_r.rolling(20,min_periods=20).std().iloc[-1],
      "es_abs_return_mean_5":_r.abs().rolling(5,min_periods=5).mean().iloc[-1],
      "es_abs_return_mean_20":_r.abs().rolling(20,min_periods=20).mean().iloc[-1]}
assert np.allclose(_vec.iloc[0].to_numpy(float),np.array([_ref[k] for k in FEATURES]),
                   rtol=1e-13,atol=1e-13)
# A deliberately excluded session must not influence previous eligible features.
_excluded=_synth.copy(); _excluded.loc[_excluded.index[-2],"target_eligible"]=False
_without=validate_session_history(_excluded,_future_date,_asof)
_e=feature_state_from_eligible_sessions(_without)
assert _e.es_return_lag2.iloc[0] == _good.session_return.iloc[-3]


def expect_rejected(call,label):
    try:
        call()
    except (ValueError,RuntimeError,AssertionError):
        return
    raise AssertionError(f"FAIL OPEN: {label}")

expect_rejected(lambda:validate_session_history(_synth,_last_date.strftime('%Y-%m-%d'),_asof),"same-day leakage")
expect_rejected(lambda:validate_session_history(_synth,_future_date,
                         (session_time_utc(_last_date,17)-pd.Timedelta(minutes=1)).isoformat()),
                "last settled source unavailable")
_bad=_synth.copy(); _bad.loc[_bad.index[-1],"coverage_reviewed"]=False
expect_rejected(lambda:validate_session_history(_bad,_future_date,_asof),"unreviewed bars")
_bad=_synth.copy(); _bad.loc[_bad.index[-1],"session_return"]=99.0
expect_rejected(lambda:validate_session_history(_bad,_future_date,_asof),"return mismatch")
_bad=_synth.copy(); _bad.loc[_bad.index[-1],"multiple_underlying_symbols"]=True
expect_rejected(lambda:validate_session_history(_bad,_future_date,_asof),"contract mixture")
print("SYNTHETIC FEATURE SEMANTICS / AS-OF / FAIL-CLOSED TESTS: PASS")

SYNTHETIC FEATURE SEMANTICS / AS-OF / FAIL-CLOSED TESTS: PASS


## 7 · Manually reviewed Arena contract (offline only)

No network access is implemented here. **You must fetch and save the genuine challenge from the official Arena interface/API separately**, and manually transcribe a normalized review file into `data/processed/es_arena_challenge_review_v0_1.json`. Preserve the full raw official challenge payload separately. Read <https://headlinearena.com/docs/quickstart> for the *current* API. The review JSON must include:

```json
{
  "challenge_id": "REAL_ID_FROM_OFFICIAL_API",
  "instrument_root": "ES",
  "contract_symbol": "ESZ6",
  "session_date": "YYYY-MM-DD",
  "session_start_et": "18:00",
  "session_end_et": "17:00",
  "dead_zone_pct": 0.30,
  "deadline_utc": "YYYY-MM-DDTHH:MM:SS+00:00",
  "information_cutoff_utc": "YYYY-MM-DDTHH:MM:SS+00:00",
  "platform_rules_reviewed": true,
  "session_history_coverage_attested": true,
  "resolution_criteria": "Exact, nonempty criteria copied from the current challenge",
  "raw_challenge_snapshot_path": "data/processed/my_saved_official_challenge_payload.json"
}
```

Do not use the example ID, symbol or dates literally. The `raw_challenge_snapshot_path` must point to a saved real payload under the project. This notebook checks the file's SHA-256 but cannot authenticate that the raw payload is genuine: the human operator must inspect it. If the live contract differs from frozen v0.1 in neutral threshold or session times, **do not force a submission**; build a new version.

A `session_history_coverage_attested` value of `true` must mean that completed, expected trading sessions since the last historical cutoff were considered — never fabricate rows. All local previews are **non-submitted** and written once.

In [9]:
def normalized_arena_review(review_path, now_utc=None):
    review_path=Path(review_path)
    review=json.loads(review_path.read_text(encoding="utf-8"))
    required={"challenge_id","instrument_root","contract_symbol","session_date",
              "session_start_et","session_end_et","dead_zone_pct","deadline_utc",
              "information_cutoff_utc","platform_rules_reviewed",
              "session_history_coverage_attested","resolution_criteria",
              "raw_challenge_snapshot_path"}
    require(required.issubset(review), f"Challenge review missing: {sorted(required-set(review))}")
    cid=str(review["challenge_id"]).strip()
    require(bool(cid) and len(cid)<=128 and re.fullmatch(r"[a-zA-Z0-9_-]+",cid) is not None,
            "Invalid challenge ID; use official ID")
    require(review["instrument_root"]=="ES","Not an ES challenge")
    require(re.fullmatch(r"ES[HMUZ][0-9]{1,2}",str(review["contract_symbol"])) is not None,
            "Unrecognized ES front-contract symbol")
    require(review["session_start_et"]=="18:00" and review["session_end_et"]=="17:00",
            "Arena session differs from frozen ES session contract")
    require(type(review["dead_zone_pct"]) in (int,float) and
            abs(float(review["dead_zone_pct"])-TARGET_BAND_PCT)<=1e-12,
            "Live neutral band differs from historical ±0.30%; do not reuse silently")
    require(review["platform_rules_reviewed"] is True and
            review["session_history_coverage_attested"] is True,
            "Manual resolution and full history coverage attestations required")
    require(len(str(review["resolution_criteria"]).strip())>=30,
            "Copy the actual resolution criteria for review")
    challenge_date=pd.Timestamp(review["session_date"])
    require(challenge_date==challenge_date.normalize() and challenge_date.tzinfo is None,
            "Invalid ET settlement-day date")
    require(challenge_date > pd.Timestamp(TRAIN_THROUGH),"No historical backtest previews")
    now = datetime.now(UTC) if now_utc is None else as_utc_timestamp(now_utc,"now_utc").to_pydatetime()
    info=as_utc_timestamp(review["information_cutoff_utc"],"information_cutoff_utc")
    deadline=as_utc_timestamp(review["deadline_utc"],"deadline_utc")
    require(info<=pd.Timestamp(now) < deadline,"Cutoff in future or submission deadline already passed")
    rawpath=Path(str(review["raw_challenge_snapshot_path"]))
    require(not rawpath.is_absolute() and '..' not in rawpath.parts,
            "Raw challenge path must be project-relative and nontraversing")
    rawpath=(PROJECT_ROOT/rawpath).resolve()
    require(rawpath.is_relative_to(PROJECT_ROOT.resolve()) and rawpath.is_file(),
            "Official challenge source snapshot missing/outside project")
    return review,challenge_date,info,deadline,sha256_file(rawpath)


def local_preview(review_file, sessions_file, deployed_model, deployment_manifest,
                  now_utc=None):
    require(deployed_model is not None and deployment_manifest is not None,
            "A VERIFIED frozen deployment artifact is required for live preview")
    review,challenge_day,cutoff,deadline,raw_hash=normalized_arena_review(review_file,now_utc)
    eligible=validate_session_history(sessions_file,challenge_day,cutoff)
    require(eligible.session_date.max() > pd.Timestamp(TRAIN_THROUGH),
            "Market data is stale: no eligible completed session after frozen 2026-09-30")
    # Require the last eligible continuous-volume symbol to match the challenge's
    # selected actual front contract; stronger matching needs live supplier audit.
    require(eligible.raw_symbol.iloc[-1]==review["contract_symbol"],
            "Underlying symbol differs from the reviewed Arena challenge")
    features=feature_state_from_eligible_sessions(eligible)
    probs=ordered_model_probabilities(deployed_model,features)[0]
    baseline=check_probability_matrix(prior_p.reshape(1,3))[0]
    packet={
        "status":"LOCAL_PREVIEW_ONLY_NOT_SUBMITTED",
        "challenge_id":review["challenge_id"],
        "challenge_review_sha256":sha256_file(review_file),
        "raw_official_payload_sha256":raw_hash,
        "session_history_sha256":sha256_file(sessions_file),
        "challenge_session_date":str(challenge_day.date()),
        "deadline_utc":deadline.isoformat(),
        "information_cutoff_utc":cutoff.isoformat(),
        "last_eligible_source_session":str(eligible.session_date.max().date()),
        "last_available_utc":as_utc_timestamp(eligible.data_available_utc.iloc[-1],
                                                "data_available_utc").isoformat(),
        "contract_symbol":review["contract_symbol"],
        "model_sha256":deployment_manifest["model_sha256"],
        "deployment_id":DEPLOYMENT_ID,
        "feature_order":list(FEATURES),
        "features":{col:float(features[col].iloc[0]) for col in FEATURES},
        "probabilities":{c:float(probs[i]) for i,c in enumerate(CLASSES)},
        "static_training_prior":{c:float(baseline[i]) for i,c in enumerate(CLASSES)},
        "argmax_direction":CLASSES[int(np.argmax(probs))],
        "argmax_confidence":float(np.max(probs)),
        "arena_request_preview":{
            "probabilities":{c:float(probs[i]) for i,c in enumerate(CLASSES)},
            "reasoning":"Frozen ES-only lagged market features; no news, no same-day data. Research forecast; historical advantage inconclusive."
        },
        "generated_at_utc":datetime.now(UTC).isoformat(),
        "submitted":False,
        "platform_response":None,
        "disclaimer":"Do not mistake this preview for an accepted scored Arena submission"
    }
    # One-prediction-per-challenge policy. Refuse changed duplicates.
    preview_dir=OUTPUT_ROOT/"manual_previews"
    preview_dir.mkdir(parents=True,exist_ok=True)
    filename=preview_dir/f"preview_{review['challenge_id']}.json"
    payload=canonical_bytes(packet)
    try:
        with filename.open('xb') as f:
            f.write(payload)
    except FileExistsError:
        raise FileExistsError("Challenge already has a preview record; do not silently replace or revise it")
    return filename,packet

print("Arena challenge validator and LOCAL-ONLY preview builder: READY")

Arena challenge validator and LOCAL-ONLY preview builder: READY


## 8 · Safety gate for optional manual live preview

`RUN_MANUAL_PREVIEW=False` is the recommended default. Set it to `True` only after creating, reviewing and archiving genuine provider session aggregates and the real Arena contract files. This cell **never sends a prediction**; you must independently submit and preserve the accepted response in a later, separately reviewed workflow.

In [10]:
# NB10_PREVIEW_GATE_SENTINEL_V0_1
if not RUN_MANUAL_PREVIEW:
    print("SKIPPED: optional live preview disabled. No file/API/network action.")
else:
    assert_fresh_jupyter_execution_through("# NB10_PREVIEW_GATE_SENTINEL_V0_1")
    require(WRITE_DEPLOYMENT_ARTIFACTS,"Preview requires verified deployment artifact export")
    require(PREVIEW_SESSIONS_PATH.is_file() and PREVIEW_CHALLENGE_PATH.is_file(),
            "Manual preview input files missing; never fabricate current market data")
    verified_model, verified_manifest = validate_saved_model(MODEL_DIR)
    path,preview_packet=local_preview(PREVIEW_CHALLENGE_PATH,PREVIEW_SESSIONS_PATH,
                                      verified_model,verified_manifest)
    print("LOCAL PREVIEW SAVED (not submitted):",path)
    display(pd.DataFrame([preview_packet["probabilities"],
                          preview_packet["static_training_prior"]],
                         index=["candidate LR","frozen static prior"]))
    print("NO Arena POST request was made. This is not a scored forecast.")

SKIPPED: optional live preview disabled. No file/API/network action.


## 9 · Local integrity checklist, Git workflow and prospective next steps

### After first clean local Run All (with both flags `False`)

From Windows Command Prompt, in the experimental branch:

```bat
cd /d C:\Users\melny\Market-Mood-Forecasting
git status -sb
git add -- "notebooks/10_es_arena_prospective_inference_v0_1.ipynb"
git diff --cached --check
git diff --cached --name-status
git commit -m "research: add ES Arena prospective deployment Notebook 10"
git push origin experiment/headline-arena-es-daily
git fetch origin
```

Check outputs and the absence of secrets / restricted source-market records before committing. Do **not** stage `data/processed/`, raw Databento inputs, or `check_databento_cost.py`. Do not merge into `main`.

Next set only `WRITE_DEPLOYMENT_ARTIFACTS=True`, save, **Kernel → Restart & Run All**. The source guard accepts that exact boolean change once Notebook 10 is committed and pushed. It then exports an immutable deployment directory. Review hashes, copy to a private backup and **do not publish fitted artifacts without a licensing/security review**.

### Before a genuine live Arena challenge

1. Acquire **only** missing finalized ES market-session data from the same provider after checking availability and estimated cost. The frozen parquet does not contain all raw `high/low/volume` fields required to reconstruct future lags.
2. Save original provider payloads and review session boundaries, actual raw contract(s), eligibility, coverage, data-availability timestamps and complete session coverage.
3. Obtain the **genuine** current Arena challenge and save its raw response. Inspect actual deadline, symbol and resolution criteria. Populate manual review JSON; never assume ±0.30% when the challenge differs.
4. Run a **local preview** by enabling `RUN_MANUAL_PREVIEW=True` after a model has been frozen, with valid reviewed input files; export a non-submitted prediction record.
5. In a **separate reviewed workflow**, submit **one full probability vector** before the platform's actual deadline, preserve the authenticated accepted response, wait for settlement and reconcile Brier locally. The submitted inference model remains frozen.
6. Keep a strictly append-only live scorecard for both the model and frozen prior, including unsuccessful/negative results. A new fit or changed feature set must be assigned a new version.

**Not supported in Notebook 10:** paid Databento requests, automatic retrieval, registration/authentication, network transmissions, Arena POST requests, automated backtest retuning, or reopening Notebook 09. These are intentionally excluded for reproducibility and safety.

Reference: [Headline Arena developer quickstart](https://headlinearena.com/docs/quickstart) and [current methodology](https://headlinearena.com/methodology). API behavior is time-dependent: recheck before live submission.

In [11]:
# Final read-only status. Never run Notebook 09's consumed evaluation here.
print("="*70)
print("NOTEBOOK 10 STATUS")
print("Upstream dataset SHA:",FROZEN_DATA_SHA256)
print("Training cutoff:",TRAIN_THROUGH,"| feature-complete rows:",len(training))
print("Same 12 predictors:",list(FEATURES))
print("New deployment estimator fit:",bool(hasattr(model,'classes_')))
print("Frozen Notebook 09 source provenance verified:",UPSTREAM_GIT_VERIFIED)
print("Frozen Notebook 09 expected/observed SHA:",FROZEN_MODEL_SOURCE_SHA256,OBSERVED_NB09_GIT_SHA256)
print("Notebook 10 source committed:",NOTEBOOK_COMMITTED)
print("Git HEAD synced to origin:",GIT_SYNCED)
print("Local artifact export enabled:",WRITE_DEPLOYMENT_ARTIFACTS)
print("Manual preview enabled:",RUN_MANUAL_PREVIEW)
print("Model artifact present:",bool(MODEL_DIR.is_dir() and (MODEL_DIR/MODEL_MANIFEST_NAME).is_file()))
print("Live submission calls: NOT IMPLEMENTED")
print("Notebook 09 final holdout: NOT TOUCHED")
print("="*70)

NOTEBOOK 10 STATUS
Upstream dataset SHA: 6787e64fe1c0c3508d68cd2d98cc554c6e5da19f320c7080ad7e34831b9b5db5
Training cutoff: 2026-09-30 | feature-complete rows: 3835
Same 12 predictors: ['es_return_lag1', 'es_return_lag2', 'es_return_lag3', 'es_return_lag5', 'es_range_pct_lag1', 'es_log_volume_lag1', 'es_return_mean_5', 'es_return_mean_20', 'es_vol_5', 'es_vol_20', 'es_abs_return_mean_5', 'es_abs_return_mean_20']
New deployment estimator fit: True
Frozen Notebook 09 source provenance verified: True
Frozen Notebook 09 expected/observed SHA: 166fe687e441b6b1f0aa74659eb6fa2cfdaed72523b1380976552f777b50ae8e 166fe687e441b6b1f0aa74659eb6fa2cfdaed72523b1380976552f777b50ae8e
Notebook 10 source committed: False
Git HEAD synced to origin: True
Local artifact export enabled: False
Manual preview enabled: False
Model artifact present: False
Live submission calls: NOT IMPLEMENTED
Notebook 09 final holdout: NOT TOUCHED
